# Apêndice 10.2: Uso de Ferramentas

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Área de Testes](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY variable from the IPython store
%store -r API_KEY

client = anthropic.Anthropic(api_key=API_KEY)

# Rewrittten to call Claude 3 Sonnet, which is generally better at tool use, and include stop_sequences
def get_completion(messages, system_prompt="", prefill="",stop_sequences=None):
    message = client.messages.create(
        model="claude-3-sonnet-20240229",
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=messages,
        stop_sequences=stop_sequences
    )
    return message.content[0].text

---

## Lição

Embora possa parecer conceitualmente complexo no início, o uso de ferramentas, também conhecido como chamada de função, é na verdade bastante simples! Você já conhece todas as habilidades necessárias para implementar o uso de ferramentas, que é realmente apenas uma combinação de substituição e encadeamento de prompts.

Em exercícios de substituição anteriores, substituímos texto em prompts. Com o uso de ferramentas, substituímos resultados de ferramentas ou funções em prompts. O Claude não pode literalmente chamar ou acessar ferramentas e funções. Em vez disso, fazemos o Claude:
1. Produzir o nome da ferramenta e os argumentos que ele quer chamar
2. Interromper qualquer geração de resposta adicional enquanto a ferramenta é chamada
3. Então repromptamos com os resultados da ferramenta anexados

Chamada de função é útil porque expande as capacidades do Claude e permite que o Claude lide com tarefas muito mais complexas e de múltiplas etapas.
Alguns exemplos de funções que você pode dar ao Claude:
- Calculadora
- Contador de palavras
- Consulta de banco de dados SQL e recuperação de dados
- API de clima

Você pode fazer o Claude usar ferramentas combinando estes dois elementos:

1. Um system prompt, no qual damos ao Claude uma explicação do conceito de uso de ferramentas, bem como uma lista descritiva detalhada das ferramentas às quais ele tem acesso
2. A lógica de controle com a qual orquestrar e executar as solicitações de uso de ferramentas do Claude

### Roteiro de uso de ferramentas

*Esta lição ensina nosso formato atual de uso de ferramentas. No entanto, estaremos atualizando e melhorando a funcionalidade de uso de ferramentas em um futuro próximo, incluindo:*
* *Um formato mais simplificado para definições e chamadas de função*
* *Tratamento de erros mais robusto e cobertura de casos extremos*
* *Integração mais estreita com o resto de nossa API*
* *Melhor confiabilidade e desempenho, especialmente para tarefas de uso de ferramentas mais complexas*

### Exemplos

Para habilitar o uso de ferramentas no Claude, começamos com o system prompt. Neste system prompt especial de uso de ferramentas, dizemos ao Claude:
* A premissa básica do uso de ferramentas e o que isso envolve
* Como o Claude pode chamar e usar as ferramentas que recebeu
* Uma lista detalhada de ferramentas às quais ele tem acesso neste cenário específico

Aqui está a primeira parte do system prompt, explicando o uso de ferramentas ao Claude. Esta parte do system prompt é generalizável em todas as instâncias de prompting do Claude para uso de ferramentas. A estrutura de chamada de ferramenta que estamos dando ao Claude (`<function_calls> [...] </function_calls>`) é uma estrutura que o Claude foi especificamente treinado para usar, então recomendamos que você mantenha isso.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

Aqui está a segunda parte do system prompt, que define as ferramentas exatas às quais o Claude tem acesso nesta situação específica. Neste exemplo, daremos ao Claude uma ferramenta de calculadora, que recebe três parâmetros: dois operandos e um operador.

Então combinamos as duas partes do system prompt.

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

Agora podemos dar ao Claude uma pergunta que requer o uso da ferramenta `calculator`. Usaremos `<function_calls>` em `stop_sequences` para detectar se e quando o Claude chama a função.

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Agora, podemos extrair os parâmetros da chamada de função do Claude e realmente executar a função em nome do Claude.

Primeiro vamos definir o código da função.

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

Então vamos extrair os parâmetros da resposta de chamada de função do Claude. Se todos os parâmetros existirem, executamos a ferramenta calculadora.

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

Agora que temos um resultado, temos que formatar adequadamente esse resultado para que quando passarmos de volta ao Claude, o Claude entenda a qual ferramenta esse resultado está relacionado. Há um formato definido para isso que o Claude foi treinado para reconhecer:
```
<function_results>
<result>
<tool_name>{TOOL_NAME}</tool_name>
<stdout>
{TOOL_RESULT}
</stdout>
</result>
</function_results>
```

Execute a célula abaixo para formatar o resultado da ferramenta acima nesta estrutura.

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

Agora tudo o que temos que fazer é enviar este resultado de volta ao Claude anexando o resultado à mesma cadeia de mensagens de antes, e estamos prontos!

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Construct the full conversation
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Print Claude's response
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- FINAL RESULT -------------")
print(final_response)

Parabéns por executar uma cadeia completa de uso de ferramentas de ponta a ponta!

Agora, e se dermos ao Claude uma pergunta que não requer o uso da ferramenta dada?

In [ ]:
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Sucesso! Como você pode ver, o Claude sabia não chamar a função quando não era necessário.

Se você quiser experimentar com os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar a [**Área de Testes**](#example-playground).

---

## Exercícios
- [Exercício 10.2.1 - SQL](#exercise-1021---SQL)

### Exercício 10.2.1 - SQL
Neste exercício, você estará escrevendo um prompt de uso de ferramentas para consultar e escrever no menor "banco de dados" do mundo. Aqui está o banco de dados inicializado, que é realmente apenas um dicionário.

In [ ]:
db = {
    "users": [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"},
        {"id": 3, "name": "Charlie", "email": "charlie@example.com"}
    ],
    "products": [
        {"id": 1, "name": "Widget", "price": 9.99},
        {"id": 2, "name": "Gadget", "price": 14.99},
        {"id": 3, "name": "Doohickey", "price": 19.99}
    ]
}

E aqui está o código para as funções que escrevem e leem do banco de dados.

In [ ]:
def get_user(user_id):
    for user in db["users"]:
        if user["id"] == user_id:
            return user
    return None

def get_product(product_id):
    for product in db["products"]:
        if product["id"] == product_id:
            return product
    return None

def add_user(name, email):
    user_id = len(db["users"]) + 1
    user = {"id": user_id, "name": name, "email": email}
    db["users"].append(user)
    return user

def add_product(name, price):
    product_id = len(db["products"]) + 1
    product = {"id": product_id, "name": name, "price": price}
    db["products"].append(product)
    return product

Para resolver o exercício, comece definindo um system prompt como `system_prompt_tools_specific_tools` acima. Certifique-se de incluir o nome e descrição de cada ferramenta, juntamente com o nome, tipo e descrição de cada parâmetro para cada função. Demos a você alguma estrutura inicial abaixo.

In [ ]:
system_prompt_tools_specific_tools_sql = """
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools_sql

Quando estiver pronto, você pode experimentar seu system prompt de definição de ferramentas nos exemplos abaixo. Apenas execute a célula abaixo!

In [ ]:
examples = [
    "Add a user to the database named Deborah.",
    "Add a product to the database named Thingo",
    "Tell me the name of User 2",
    "Tell me the name of Product 3"
]

for example in examples:
    message = {
        "role": "user",
        "content": example
    }

    # Get & print Claude's response
    function_calling_response = get_completion([message], system_prompt=system_prompt, stop_sequences=stop_sequences)
    print(example, "\n----------\n\n", function_calling_response, "\n*********\n*********\n*********\n\n")

Se você fez certo, as mensagens de chamada de função devem chamar as funções `add_user`, `add_product`, `get_user` e `get_product` corretamente.

Para crédito extra, adicione algumas células de código e escreva código de análise de parâmetros. Então chame as funções com os parâmetros que o Claude lhe dá para ver o estado do "banco de dados" após a chamada.

❓ Se você quiser ver uma possível solução, execute a célula abaixo!

In [ ]:
from hints import exercise_10_2_1_solution; print(exercise_10_2_1_solution)

### Parabéns!

Parabéns por aprender o uso de ferramentas e chamada de função! Vá para a última seção do apêndice se você quiser aprender mais sobre busca e RAG.

---

## Área de Testes

Esta é uma área para você experimentar livremente com os exemplos de prompt mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Construct the full conversation
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Print Claude's response
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- FINAL RESULT -------------")
print(final_response)

In [ ]:
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)